In [1]:
# Setup — load train.csv, train_series.csv, finding_columns, gold + extracted labels
import pandas as pd
import numpy as np
import pydicom
import cv2
import os

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv(f"{data_dir}/train.csv")
train_series = pd.read_csv(f"{data_dir}/train_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

gold_data = train[train['Fracture'].notna()]
gold_labels = gold_data[['StudyInstanceUID'] + finding_columns].copy()

report_label = pd.read_csv("/kaggle/input/datasets/mayankptdr/report-labels/extracted_labels.csv")

print(f"Gold: {gold_labels.shape}, Report-extracted: {report_label.shape}")

Gold: (58, 13), Report-extracted: (4349, 13)


In [ ]:
# Updated finding definitions + prompt builder — 3-state system (1=confirmed, 0=explicitly-absent, null=not-mentioned)
finding_definitions = """
- ACL: anterior cruciate ligament tear/rupture/injury (any grade). Multilingual terms: "LCA" (Spanish), "VKB" (German), "voorste kruisband" (Dutch), "Ön çapraz bağ" (Turkish), "πρόσθιου χιαστού" (Greek), "предната кръстна връзка" (Bulgarian)
- MCL: medial collateral ligament injury/sprain/tear (any grade). Terms: "LCM" (Spanish), "medial kollateral ligaman" (Turkish)
- Medial Meniscus: medial meniscus tear (radial, bucket-handle, degenerative, horizontal, longitudinal). Terms: "menisco interno/medial" (Spanish), "medial menisküs" (Turkish), "Innenmeniskus" (German)
- Lateral Meniscus: lateral meniscus tear (any type). Terms: "menisco lateral" (Spanish), "lateral menisküs" (Turkish), "Außenmeniskus" (German)
- Medial OA: osteoarthritis/chondrosis/cartilage loss of the medial tibiofemoral compartment. Terms: "condropatía", "OA femorotibial medial" (Spanish), "kraakbeenlijden" (Dutch), "hondromalacija" (Croatian)
- Lateral OA: same as above, lateral tibiofemoral compartment
- PF OA: patellofemoral chondrosis/chondromalacia/cartilage loss. Terms: "condropatía rotuliana" (Spanish)
- Effusion: joint/knee effusion, joint fluid, hemarthrosis. Terms: "derrame articular" (Spanish), "izljev" (Croatian), "Gelenkerguss" (German)
- Synovitis: synovial thickening/hypertrophy/inflammation. Terms: "sinovitis" (Spanish), "sinovit"/"proliferaciju sinovije" (Croatian)
- Baker's: Baker's cyst / popliteal cyst (same thing). Terms: "quiste poplíteo" (Spanish), "Baker cista" (Croatian)
- Contusion: bone contusion/bruise/marrow edema from trauma, at a site SEPARATE from any fracture site. If the edema is described as directly surrounding/adjacent to a fracture (same bone/same location), do NOT count it as a separate Contusion — only count Contusion if a distinct bone bruise is mentioned at a DIFFERENT location than the fracture, or if there is no fracture at all.
- Fracture: any bone fracture (avulsion, impaction, subchondral, stress, osteochondral). Note: Spanish "fractura" = fracture, but "rotura" ALONE (without "fractura") usually refers to a soft-tissue/ligament/meniscus tear, not bone.
"""

few_shot_examples = """
Example 1 (Spanish, multiple findings explicitly confirmed, rest explicitly normal):
Report: "Rotura del LCA. Rotura parcial del LCM. Rotura de menisco interno y lateral. Derrame. Contusiones óseas femorotibiales. Ligamentos cruzados posteriores y colaterales laterales sin alteraciones. No hay quistes poplíteos."
Output: {"ACL":1,"MCL":1,"Medial Meniscus":1,"Lateral Meniscus":1,"Medial OA":null,"Lateral OA":null,"PF OA":null,"Effusion":1,"Synovitis":null,"Baker's":0,"Contusion":1,"Fracture":null}
Reasoning: Baker's=0 because "no quistes poplíteos" (no popliteal cysts) is an EXPLICIT negative statement. Medial OA, Lateral OA, PF OA, Synovitis, Fracture are null because the report never mentions them at all (not even to say they're normal).

Example 2 (English, mostly normal exam with explicit negatives):
Report: "ACL is intact. PCL is preserved. The MCL is intact. Medial meniscus is not torn. Lateral meniscus is normal. No fracture is seen. No joint effusion."
Output: {"ACL":0,"MCL":0,"Medial Meniscus":0,"Lateral Meniscus":0,"Medial OA":null,"Lateral OA":null,"PF OA":null,"Effusion":0,"Synovitis":null,"Baker's":null,"Contusion":null,"Fracture":0}
Reasoning: Every finding here has an EXPLICIT normal statement gets 0. OA, PF OA, Synovitis, Baker's are never mentioned at all, so null.

Example 3 (fracture with edema at a DIFFERENT site = both Fracture and Contusion confirmed):
Report: "МР данни за фрактура на тибията... Наличие на костно-мозъчен едем около фрактурата. Контузионен костно-мозъчен едем се визуализира и в латералния феморален кондил."
Output: {"ACL":null,"MCL":null,"Medial Meniscus":null,"Lateral Meniscus":null,"Medial OA":null,"Lateral OA":null,"PF OA":null,"Effusion":null,"Synovitis":null,"Baker's":null,"Contusion":1,"Fracture":1}
"""

def build_prompt(report_text):
    return f"""You are a radiology report labeler. Read the following knee MRI report, which may be written in any language (English, Spanish, German, Dutch, Greek, Bulgarian, Turkish, etc.).

Use these definitions and multilingual synonyms:
{finding_definitions}

Study these examples of correct labeling:
{few_shot_examples}

Now label the following report. Output ONLY a JSON object with these 12 keys: ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.

CRITICAL RULE — use exactly these three states:
- 1 if the finding is explicitly stated as present/positive/abnormal
- 0 ONLY if the report explicitly states this specific finding is normal/absent/intact (an explicit negative statement about THIS finding)
- null if the finding is not mentioned anywhere in the report (no statement either way)

Do NOT default to 0 when something is simply not discussed — use null for that. Only use 0 when there is a clear, explicit statement that this particular structure/finding is normal.

Do not include any text other than the JSON object.

Report:
{report_text}
"""

print("Prompt builder ready (3-state version)")

In [ ]:
#  Gemini API client setup
from google import genai

client = genai.Client(api_key="AQ.Ab8RN6Igjuo3NVm_3Gax5dXG3I0A8igNks09b2CtOy55JpkImg")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Test connection, reply with OK"
)
print(response.text)

In [ ]:
# 3-state prompt on 58 gold-labeled studies
import json
import time

results = []

for idx, row in gold_data.iterrows():
    report_text = row['Report']
    
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=build_prompt(report_text)
        )
        cleaned = response.text.strip().replace("```json", "").replace("```", "").strip()
        predicted = json.loads(cleaned)
    except Exception as e:
        predicted = {col: None for col in finding_columns}
        print(f"Error on {row['StudyInstanceUID']}: {e}")
    
    actual = row[finding_columns].to_dict()
    results.append({'study_id': row['StudyInstanceUID'], 'predicted': predicted, 'actual': actual})
    time.sleep(0.5)

print(f"Processed {len(results)} studies")

In [ ]:
#Accuracy-calculation — 3-state-aware 
correct_counts = {col: 0 for col in finding_columns}
total_counts = {col: 0 for col in finding_columns}
null_counts = {col: 0 for col in finding_columns}

for r in results:
    for col in finding_columns:
        pred = r['predicted'].get(col)
        actual = r['actual'].get(col)
        if actual is not None:
            actual = int(actual)
        
        if pred is None:
            null_counts[col] += 1
            continue  
        
        total_counts[col] += 1
        if pred == actual:
            correct_counts[col] += 1

print("Per-finding accuracy (sirf-confident-predictions-pe) + null-rate:")
for col in finding_columns:
    acc = correct_counts[col] / total_counts[col] * 100 if total_counts[col] > 0 else 0
    null_rate = null_counts[col] / 58 * 100
    print(f"{col}: acc={acc:.1f}% ({correct_counts[col]}/{total_counts[col]}), null-rate={null_rate:.1f}%")

In [ ]:
# Cell: Bulk re-extraction — 3-state prompt, resumable
output_file_v2 = "extracted_labels_v2.csv"

if os.path.exists(output_file_v2):
    done_df = pd.read_csv(output_file_v2)
    done_ids = set(done_df['StudyInstanceUID'])
    print(f"Already done: {len(done_ids)}")
else:
    done_ids = set()

remaining = train[~train['StudyInstanceUID'].isin(gold_data['StudyInstanceUID'])]
remaining = remaining[~remaining['StudyInstanceUID'].isin(done_ids)]
print(f"Remaining to process: {len(remaining)}")

In [ ]:
# Bulk extraction loop — resumable, 3-state labels
for idx, row in remaining.iterrows():
    study_id = row['StudyInstanceUID']
    report_text = row['Report']
    
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=build_prompt(report_text)
        )
        cleaned = response.text.strip().replace("```json", "").replace("```", "").strip()
        predicted = json.loads(cleaned)
        
        new_row = pd.DataFrame([{**{"StudyInstanceUID": study_id}, **predicted}])
        new_row.to_csv(output_file_v2, mode='a', header=not os.path.exists(output_file_v2), index=False)
        
        print(f"Done: {study_id}")
        time.sleep(0.1)
        
    except Exception as e:
        error_str = str(e)
        
        if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
            print(f"Rate limit hit at {study_id}. Stopping for now — resume later.")
            break
        
        elif "503" in error_str or "UNAVAILABLE" in error_str:
            print(f"Server busy on {study_id}, retrying once...")
            time.sleep(2)
            try:
                response = client.models.generate_content(
                    model="gemini-3.5-flash-lite",
                    contents=build_prompt(report_text)
                )
                cleaned = response.text.strip().replace("```json", "").replace("```", "").strip()
                predicted = json.loads(cleaned)
                new_row = pd.DataFrame([{**{"StudyInstanceUID": study_id}, **predicted}])
                new_row.to_csv(output_file_v2, mode='a', header=not os.path.exists(output_file_v2), index=False)
                print(f"Done (retry): {study_id}")
            except Exception as retry_e:
                print(f"Retry failed on {study_id}: {retry_e}")
            continue
        
        else:
            print(f"Other error on {study_id}: {error_str}")
            continue

print(f"\nLoop finished. Total processed: {len(pd.read_csv(output_file_v2))}")

In [2]:
# Restore extracted_labels_v2.csv
import shutil
old_path = '/kaggle/input/notebooks/mayankptdr/dinov2-v2/extracted_labels_v2.csv'
shutil.copy(old_path, 'extracted_labels_v2.csv')

v2_labels = pd.read_csv("extracted_labels_v2.csv")
print(v2_labels.shape)

(4349, 13)


In [3]:
# Cell: Combine gold + v2 labels
all_labels_v2 = pd.concat([gold_labels, v2_labels], ignore_index=True)


for col in finding_columns:
    all_labels_v2[f"{col}_mask"] = all_labels_v2[col].notna().astype(float)


all_labels_v2[finding_columns] = all_labels_v2[finding_columns].fillna(0)

print(all_labels_v2.shape)
print(all_labels_v2[[c for c in all_labels_v2.columns if '_mask' in c]].mean())

(4407, 25)
ACL_mask                 0.911958
MCL_mask                 0.892217
Medial Meniscus_mask     0.943499
Lateral Meniscus_mask    0.903109
Medial OA_mask           0.698888
Lateral OA_mask          0.617200
PF OA_mask               0.773769
Effusion_mask            0.907420
Synovitis_mask           0.148173
Baker's_mask             0.513047
Contusion_mask           0.610166
Fracture_mask            0.563876
dtype: float64


In [4]:
# 3-channel slice grouping
def select_slice_indices_grouped(total_slices, n_groups=16, band=(0.12, 0.88)):
    """
    n_groups wale 3-channel-groups return karta hai.
    Har group = (center_idx-1, center_idx, center_idx+1) — teen-adjacent-slices
    """
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    center_indices = np.linspace(start, end - 1, n_groups).astype(int)
    
    groups = []
    for center in center_indices:
        prev_idx = max(0, center - 1)
        next_idx = min(total_slices - 1, center + 1)
        groups.append((prev_idx, center, next_idx))
    
    return groups

# testing
test_groups = select_slice_indices_grouped(total_slices=22, n_groups=16)
print(test_groups[:5])

[(np.int64(1), np.int64(2), np.int64(3)), (np.int64(2), np.int64(3), np.int64(4)), (np.int64(3), np.int64(4), np.int64(5)), (np.int64(4), np.int64(5), np.int64(6)), (np.int64(5), np.int64(6), np.int64(7))]


In [5]:

try:
    print(crop_and_resize)
    print("Already defined")
except NameError:
    print("NOT defined, need to add")

NOT defined, need to add


In [6]:
# Core helper functions — series-selection, mm-crop, resize
import cv2

SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, series_df):
    study_series = series_df[series_df['StudyInstanceUID'] == study_id]
    slots = {}
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    return slots

def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    row_spacing, col_spacing = pixel_spacing
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    half_h, half_w = crop_h // 2, crop_w // 2
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    return pixel_array[top:bottom, left:right]

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    resized = cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)
    return resized

print("Helper functions ready")

Helper functions ready


In [7]:
# Load 3-channel grouped slice (RGB = 3 adjacent slices), with mm-crop-resize
def load_rgb_group(slices_sorted, group_indices, pixel_spacing, crop_mm=130.0, final_size=224):
    channels = []
    for idx in group_indices:
        arr = slices_sorted[idx].pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
        cr = crop_and_resize(arr, pixel_spacing, crop_mm, final_size)
        channels.append(cr)
    
    rgb_image = np.stack(channels, axis=-1)  # (H, W, 3) — R,G,B = 3-adjacent-slices
    return rgb_image

print("RGB-group loader ready")

RGB-group loader ready


In [8]:
# study-loader — 5-slots, 16 RGB-groups-per-slot, physical-slice-ordering
def load_study_v2(study_id, series_df, data_dir_path, n_groups=16, crop_mm=130.0, final_size=224):
    slot_map = select_series_5slots(study_id, series_df)
    
    slot_volumes = []
    slot_masks = []
    
    for plane, fs in SLOTS:
        key = f"{plane}_{fs}"
        series_id = slot_map[key]
        
        if series_id is None:
            vol = np.zeros((n_groups, final_size, final_size, 3), dtype=np.float32)
            mask = 0.0
        else:
            folder = f"{data_dir_path}/train_series/{study_id}/{series_id}"
            dcm_files = os.listdir(folder)
            slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
            
            # Physical-slice-ordering: ImagePositionPatient ka projection slice-normal pe
            def get_position_key(d):
                try:
                    orientation = [float(x) for x in d.ImageOrientationPatient]
                    row_vec = np.array(orientation[:3])
                    col_vec = np.array(orientation[3:])
                    normal = np.cross(row_vec, col_vec)
                    position = np.array([float(x) for x in d.ImagePositionPatient])
                    return np.dot(position, normal)
                except Exception:
                    return int(d.InstanceNumber)  # fallback agar tags missing hon
            
            slices_sorted = sorted(slices, key=get_position_key)
            total_slices = len(slices_sorted)
            pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
            
            groups = select_slice_indices_grouped(total_slices, n_groups)
            
            rgb_groups = []
            for group_indices in groups:
                rgb = load_rgb_group(slices_sorted, group_indices, pixel_spacing, crop_mm, final_size)
                rgb_groups.append(rgb)
            
            vol = np.stack(rgb_groups).astype(np.float32)  # (n_groups, H, W, 3)
            mask = 1.0
        
        slot_volumes.append(vol)
        slot_masks.append(mask)
    
    slots_array = np.stack(slot_volumes)  # (5, n_groups, H, W, 3)
    mask_array = np.array(slot_masks, dtype=np.float32)
    
    return slots_array, mask_array

print("Full v2 study-loader ready")

Full v2 study-loader ready


In [9]:
# Testing on 1 study
test_study_id = all_labels_v2['StudyInstanceUID'].iloc[0]
slots_arr, mask_arr = load_study_v2(test_study_id, train_series, data_dir)
print(f"Slots shape: {slots_arr.shape}")
print(f"Mask: {mask_arr}")

Slots shape: (5, 16, 224, 224, 3)
Mask: [1. 1. 1. 1. 1.]


In [10]:
# Verify all 10 v2-cache datasets, count total files
base = '/kaggle/input/datasets/mayankptdr'
print(os.listdir(base))

['v2-labels', 'v2-rsna-4', 'report-labels', 'v2-rsna-7', 'v2-rsna-8', 'v2-rsna-10', 'v2-rsna-1', 'v2-rsna-5', 'v2-rsna-3', 'v2-rsna-9', 'v2-rsna-2', 'v2-rsna-6']


In [11]:
# Verify v2-cache paths and total file count across all 10 chunks
cache_dirs_v2 = []
total_files = 0

for i in range(1, 11):
    name = f'v2-rsna-{i}'
    path = f"{base}/{name}/cached_v2"
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"{name}: {count} files")
        cache_dirs_v2.append(path)
        total_files += count
    else:
        print(f"{name}: path not found, checking structure...")
        print(os.listdir(f"{base}/{name}"))

print(f"\nTotal cached studies: {total_files}")

v2-rsna-1: 440 files
v2-rsna-2: 440 files
v2-rsna-3: 440 files
v2-rsna-4: 440 files
v2-rsna-5: 440 files
v2-rsna-6: 440 files
v2-rsna-7: 440 files
v2-rsna-8: 440 files
v2-rsna-9: 440 files
v2-rsna-10: 447 files

Total cached studies: 4407


In [12]:
import torch
from torch.utils.data import Dataset

class KneeMRIDatasetV2(Dataset):
    def __init__(self, labels_df, cache_dirs):
        self.labels_df = labels_df.reset_index(drop=True)
        self.cache_dirs = cache_dirs
        self.finding_columns = finding_columns
        
        self.valid_indices = []
        for idx, row in self.labels_df.iterrows():
            if self._find_cache_path(row['StudyInstanceUID']) is not None:
                self.valid_indices.append(idx)
        
        print(f"Valid studies (cache found): {len(self.valid_indices)} / {len(self.labels_df)}")
    
    def _find_cache_path(self, study_id):
        for d in self.cache_dirs:
            path = f"{d}/{study_id}.npz"
            if os.path.exists(path):
                return path
        return None
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, i):
        idx = self.valid_indices[i]
        row = self.labels_df.iloc[idx]
        study_id = row['StudyInstanceUID']
        
        cache_path = self._find_cache_path(study_id)
        data = np.load(cache_path)
        
        # slots: (5, 16, 224, 224, 3) float16 -> convert to (5, 16, 3, 224, 224) float32 for PyTorch
        slots_raw = data['slots'].astype(np.float32)
        slots = torch.tensor(slots_raw).permute(0, 1, 4, 2, 3) 
        
        slot_mask = torch.tensor(data['mask'], dtype=torch.float32)
        
        labels = torch.tensor(row[self.finding_columns].values.astype(float), dtype=torch.float32)
        label_mask = torch.tensor(row[[f"{c}_mask" for c in self.finding_columns]].values.astype(float), dtype=torch.float32)
        
        return {'slots': slots, 'slot_mask': slot_mask, 'labels': labels, 'label_mask': label_mask, 'study_id': study_id}

dataset_v2 = KneeMRIDatasetV2(all_labels_v2, cache_dirs_v2)
sample = dataset_v2[0]
print(sample['slots'].shape, sample['slot_mask'], sample['labels'], sample['label_mask'])

Valid studies (cache found): 4407 / 4407
torch.Size([5, 16, 3, 224, 224]) tensor([1., 1., 1., 1., 1.]) tensor([0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.]) tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [13]:
import torch.nn as nn
from transformers import Dinov2Model

class SliceAttentionPool(nn.Module):
    def __init__(self, dim=384):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (x * weights).sum(dim=1)

class DINOv2KneeModelV2(nn.Module):
    def __init__(self, dinov2_path, num_findings=12, n_slots=5, n_groups=16):
        super().__init__()
        self.dinov2 = Dinov2Model.from_pretrained(dinov2_path)
        self.dim = self.dinov2.config.hidden_size
        self.n_slots = n_slots
        self.n_groups = n_groups
        self.slice_pool = SliceAttentionPool(self.dim)
        self.slot_attn = nn.Sequential(nn.Linear(self.dim, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.Linear(self.dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_findings))
    
    def forward(self, slots, slot_mask):
        batch_size = slots.shape[0]
        # slots already (batch, 5, 16, 3, 224, 224) — no repeat needed, already RGB
        x = slots.view(batch_size * self.n_slots * self.n_groups, 3, 224, 224)
        features = self.dinov2(pixel_values=x).pooler_output
        features = features.view(batch_size, self.n_slots, self.n_groups, self.dim)
        
        slot_features = torch.stack([self.slice_pool(features[:, s, :, :]) for s in range(self.n_slots)], dim=1)
        
        slot_weights = self.slot_attn(slot_features).squeeze(-1)
        slot_weights = slot_weights.masked_fill(slot_mask == 0, float('-inf'))
        slot_weights = torch.softmax(slot_weights, dim=1).unsqueeze(-1)
        
        final_feature = (slot_features * slot_weights).sum(dim=1)
        return self.head(final_feature)

print("Model V2 ready")

Model V2 ready


In [14]:
# testing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dinov2_path = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
model = DINOv2KneeModelV2(dinov2_path).to(device)

from torch.utils.data import DataLoader
loader = DataLoader(dataset_v2, batch_size=1, shuffle=True)
batch = next(iter(loader))

slots = batch['slots'].to(device)
slot_mask = batch['slot_mask'].to(device)

output = model(slots, slot_mask)
print(output.shape)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

torch.Size([1, 12])


In [15]:
def masked_bce_loss(outputs, labels, label_mask, pos_weight):
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')
    raw_loss = loss_fn(outputs, labels)  # (batch, 12)
    masked_loss = raw_loss * label_mask  # null-wale-findings-ka-loss-zero-ho-jaata-hai
    
    # average sirf un-cells-pe jo-actually-masked-nahi-hain
    denom = label_mask.sum()
    if denom == 0:
        return raw_loss.sum() * 0  # safety: agar-poora-batch-hi-masked-hai
    return masked_loss.sum() / denom

print("Masked loss ready")

Masked loss ready


In [16]:
pos_counts = np.zeros(12)
neg_counts = np.zeros(12)

for i, col in enumerate(finding_columns):
    known = all_labels_v2[f"{col}_mask"] == 1
    pos_counts[i] = (all_labels_v2.loc[known, col] == 1).sum()
    neg_counts[i] = (all_labels_v2.loc[known, col] == 0).sum()

pos_weights_v2 = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32)
print(dict(zip(finding_columns, pos_weights_v2.numpy().round(2))))

{'ACL': np.float32(3.73), 'MCL': np.float32(5.1), 'Medial Meniscus': np.float32(1.18), 'Lateral Meniscus': np.float32(4.21), 'Medial OA': np.float32(0.89), 'Lateral OA': np.float32(1.68), 'PF OA': np.float32(0.74), 'Effusion': np.float32(0.51), 'Synovitis': np.float32(0.16), "Baker's": np.float32(1.08), 'Contusion': np.float32(1.25), 'Fracture': np.float32(6.62)}


In [17]:
!pip install -q iterative-stratification

In [18]:
# fold split on valid v2-studies (study-level, multilabel-stratified)
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

valid_labels_v2 = all_labels_v2.iloc[dataset_v2.valid_indices].reset_index(drop=True)
X = valid_labels_v2['StudyInstanceUID'].values
y = valid_labels_v2[finding_columns].values

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_v2 = []
for fold_num, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    folds_v2.append({'train_idx': train_idx, 'val_idx': val_idx})
    print(f"Fold {fold_num}: train={len(train_idx)}, val={len(val_idx)}")

Fold 0: train=3525, val=882
Fold 1: train=3526, val=881
Fold 2: train=3525, val=882
Fold 3: train=3526, val=881
Fold 4: train=3526, val=881


In [19]:
# Cell: Restore checkpoints_v2 and fold_results_v2.json from previous session
import shutil
import os

old_path = '/kaggle/input/notebooks/mayankptdr/dinov2-v2'

checkpoint_dir_v2 = "/kaggle/working/checkpoints_v2"
os.makedirs(checkpoint_dir_v2, exist_ok=True)

old_ckpt_dir = f"{old_path}/checkpoints_v2"
for f in os.listdir(old_ckpt_dir):
    shutil.copy(f"{old_ckpt_dir}/{f}", f"{checkpoint_dir_v2}/{f}")
print("Checkpoints restored:", os.listdir(checkpoint_dir_v2))

old_results = f"{old_path}/fold_results_v2.json"
if os.path.exists(old_results):
    shutil.copy(old_results, "/kaggle/working/fold_results_v2.json")
    print("fold_results_v2.json restored")
else:
    print("No fold_results_v2.json found (fold-0 abhi complete-nahi-hua-tha, yeh-expected-hai)")

Checkpoints restored: ['fold0_latest.pt', 'fold3_latest.pt', 'fold3_best.pt', 'fold4_best.pt', 'fold4_latest.pt', 'fold2_latest.pt', 'fold2_best.pt', 'fold0_best.pt', 'fold1_latest.pt', 'fold1_best.pt']
fold_results_v2.json restored


In [20]:
# Cell: Check restored fold_results and latest checkpoint epochs
import json
with open('/kaggle/working/fold_results_v2.json') as f:
    print(json.load(f))

import torch
for f in ['fold0_latest.pt', 'fold1_latest.pt']:
    ckpt = torch.load(f"/kaggle/working/checkpoints_v2/{f}", map_location='cpu', weights_only=False)
    print(f"{f}: epoch={ckpt['epoch']}, best_auroc={ckpt['best_auroc']:.4f}")

{'0': 0.7753236148521441, '1': 0.8051157166992576, '2': 0.7669831285066806, '3': 0.7763749746863043}
fold0_latest.pt: epoch=7, best_auroc=0.7753
fold1_latest.pt: epoch=7, best_auroc=0.8051


In [21]:
# Cell: Training function V2 
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score
import json
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_dir_v2 = "/kaggle/working/checkpoints_v2"
os.makedirs(checkpoint_dir_v2, exist_ok=True)

def train_one_fold_v2(fold_num, train_idx, val_idx, epochs=8, batch_size=1, lr=1e-4, accumulation_steps=4):
    train_subset = Subset(dataset_v2, train_idx)
    val_subset = Subset(dataset_v2, val_idx)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    model = DINOv2KneeModelV2(dinov2_path, num_findings=12).to(device)
    
    for param in model.dinov2.parameters():
        param.requires_grad = False
    for param in model.dinov2.encoder.layer[-2:].parameters():
        param.requires_grad = True
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    checkpoint_path = f"{checkpoint_dir_v2}/fold{fold_num}_latest.pt"
    best_path = f"{checkpoint_dir_v2}/fold{fold_num}_best.pt"
    start_epoch, best_auroc = 0, 0
    
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        start_epoch = ckpt['epoch'] + 1
        best_auroc = ckpt['best_auroc']
        print(f"Resumed fold {fold_num} from epoch {start_epoch}, best_auroc_so_far={best_auroc:.4f}")
        del ckpt
    
    for epoch in range(start_epoch, epochs):
        model.train()
        train_loss = 0
        nan_skips = 0
        optimizer.zero_grad()
        
        for step, batch in enumerate(train_loader):
            slots = batch['slots'].to(device)
            slot_mask = batch['slot_mask'].to(device)
            labels = batch['labels'].to(device)
            label_mask = batch['label_mask'].to(device)
            
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model(slots, slot_mask)
                loss = masked_bce_loss(outputs, labels, label_mask, pos_weights_v2.to(device)) / accumulation_steps
            
            if torch.isnan(loss) or torch.isinf(loss):
                nan_skips += 1
                optimizer.zero_grad()
                continue
            
            loss.backward()
            
            if (step + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
            
            if (step + 1) % 200 == 0:
                print(f"    step {step+1}/{len(train_loader)}, running_loss={train_loss/(step+1):.4f}")
            
            # === per-step chhoti-cleanup, variables-jo-ab-zaroorat-nahi ===
            del slots, slot_mask, labels, label_mask, outputs, loss
        
        if nan_skips > 0:
            print(f"  (skipped {nan_skips} NaN/Inf batches this epoch)")
        
        model.eval()
        all_preds, all_labels_val, all_masks_val = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                slots = batch['slots'].to(device)
                slot_mask = batch['slot_mask'].to(device)
                labels = batch['labels'].to(device)
                label_mask = batch['label_mask'].to(device)
                
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    outputs = torch.sigmoid(model(slots, slot_mask))
                all_preds.append(outputs.float().cpu().numpy())
                all_labels_val.append(labels.cpu().numpy())
                all_masks_val.append(label_mask.cpu().numpy())
                del slots, slot_mask, labels, label_mask, outputs
        
        all_preds = np.concatenate(all_preds)
        all_labels_val = np.concatenate(all_labels_val)
        all_masks_val = np.concatenate(all_masks_val)
        
        aurocs = []
        for i in range(12):
            known_idx = all_masks_val[:, i] == 1
            if known_idx.sum() < 2:
                aurocs.append(0.5)
                continue
            try:
                aurocs.append(roc_auc_score(all_labels_val[known_idx, i], all_preds[known_idx, i]))
            except ValueError:
                aurocs.append(0.5)
        
        macro_auroc = np.mean(aurocs)
        avg_loss = train_loss / len(train_loader)
        print(f"Fold {fold_num} | Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f} | Val AUROC: {macro_auroc:.4f}")
        
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(), 'best_auroc': max(best_auroc, macro_auroc)},
                   checkpoint_path)
        
        if macro_auroc > best_auroc:
            best_auroc = macro_auroc
            torch.save({'model_state': model.state_dict(), 'epoch': epoch,
                        'auroc': macro_auroc, 'per_finding_auroc': dict(zip(finding_columns, aurocs))}, best_path)
            print(f"  --> New best! Saved to {best_path}")
        
        # === EPOCH-END CLEANUP: har-epoch-complete-hone-ke-baad RAM/VRAM-clear ===
        del all_preds, all_labels_val, all_masks_val
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  (memory cleared after epoch {epoch+1})\n")
    
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_auroc

print("Training function V2 ready (with per-epoch + per-step cleanup)")

Training function V2 ready (with per-epoch + per-step cleanup)


In [22]:
# Outer loop
import time
from datetime import timedelta

fold_results_path_v2 = "/kaggle/working/fold_results_v2.json"

if os.path.exists(fold_results_path_v2):
    with open(fold_results_path_v2) as f:
        fold_results_v2 = json.load(f)
    fold_results_v2 = {int(k): v for k, v in fold_results_v2.items()}
else:
    fold_results_v2 = {}

print(f"Already-complete-folds: {list(fold_results_v2.keys())}")

overall_start = time.time()
completed_count = len(fold_results_v2)
remaining_folds = [f for f in range(5) if f not in fold_results_v2]
print(f"Remaining folds to train: {remaining_folds}\n")

for fold_num in range(5):
    if fold_num in fold_results_v2:
        print(f"Fold {fold_num} already complete (AUROC={fold_results_v2[fold_num]:.4f}), skipping\n")
        continue
    
    fold_start = time.time()
    print(f"{'='*50}")
    print(f"Starting Fold {fold_num}  [{completed_count}/5 folds done so far]")
    print(f"Elapsed so far: {timedelta(seconds=int(time.time() - overall_start))}")
    print(f"{'='*50}")
    
    best_auroc = train_one_fold_v2(
        fold_num=fold_num,
        train_idx=folds_v2[fold_num]['train_idx'],
        val_idx=folds_v2[fold_num]['val_idx'],
        epochs=8,
        batch_size=1
    )
    
    fold_results_v2[fold_num] = best_auroc
    with open(fold_results_path_v2, 'w') as f:
        json.dump(fold_results_v2, f)
    
    fold_time = time.time() - fold_start
    completed_count += 1
    
    avg_time_per_fold = (time.time() - overall_start) / completed_count if completed_count > 0 else fold_time
    folds_left = 5 - completed_count
    eta = avg_time_per_fold * folds_left
    
    print(f"\nFold {fold_num} complete. Best AUROC: {best_auroc:.4f}")
    print(f"Time taken for this fold: {timedelta(seconds=int(fold_time))}")
    print(f"Progress: {completed_count}/5 folds done")
    print(f"Estimated time remaining: {timedelta(seconds=int(eta))}")
    print(f"{'='*50}\n")

total_time = time.time() - overall_start
print(f"\nAll done: {fold_results_v2}")
if fold_results_v2:
    print(f"Average AUROC: {np.mean(list(fold_results_v2.values())):.4f}")
print(f"Total session time: {timedelta(seconds=int(total_time))}")

Already-complete-folds: [0, 1, 2, 3]
Remaining folds to train: [4]

Fold 0 already complete (AUROC=0.7753), skipping

Fold 1 already complete (AUROC=0.8051), skipping

Fold 2 already complete (AUROC=0.7670), skipping

Fold 3 already complete (AUROC=0.7764), skipping

Starting Fold 4  [4/5 folds done so far]
Elapsed so far: 0:00:00


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Resumed fold 4 from epoch 1, best_auroc_so_far=0.5260
    step 200/3526, running_loss=0.8292
    step 400/3526, running_loss=0.8513
    step 600/3526, running_loss=0.8438
    step 800/3526, running_loss=0.8480
    step 1000/3526, running_loss=0.8506
    step 1200/3526, running_loss=0.8560
    step 1400/3526, running_loss=0.8542
    step 1600/3526, running_loss=0.8555
    step 1800/3526, running_loss=0.8541
    step 2000/3526, running_loss=0.8560
    step 2200/3526, running_loss=0.8545
    step 2400/3526, running_loss=0.8525
    step 2600/3526, running_loss=0.8511
    step 2800/3526, running_loss=0.8497
    step 3000/3526, running_loss=0.8479
    step 3200/3526, running_loss=0.8472
    step 3400/3526, running_loss=0.8438
    step 200/3526, running_loss=0.8577
    step 400/3526, running_loss=0.8233
    step 600/3526, running_loss=0.8027
    step 800/3526, running_loss=0.7857
    step 1000/3526, running_loss=0.7775
    step 1200/3526, running_loss=0.7910
    step 1400/3526, running_loss=0